In [1]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import direction_utils as utils
import direction_learning_utils as train_utils


In [2]:
class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=27, Samples=2500, dropoutRate=0.5, 
                 kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
        super(EEGNet, self).__init__()
        
        # Handle dropout type
        if dropoutType == 'SpatialDropout2D':
            self.dropout = nn.Dropout2d(dropoutRate)
        elif dropoutType == 'Dropout':
            self.dropout = nn.Dropout(dropoutRate)
        else:
            raise ValueError('dropoutType must be one of SpatialDropout2D or Dropout.')

        # Block 1
        self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwiseConv = nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(F1*D)
        self.pool1 = nn.AvgPool2d((1, 4))

        # Block 2
        self.separableConv = nn.Conv2d(F1*D, F2, (1, 16), padding='same', bias=False)
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))

        # Flatten and Dense
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(F2 * (Samples // (4 * 8)), nb_classes)
        self.norm_constraint = nn.utils.weight_norm(self.dense)

    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.depthwiseConv(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.pool1(x)
        x = self.dropout(x)

        # Block 2
        x = self.separableConv(x)
        x = self.batchnorm3(x)
        x = F.elu(x)
        x = self.pool2(x)
        x = self.dropout(x)

        # Flatten and Dense
        x = self.flatten(x)
        x = self.dense(x)
        return F.softmax(x, dim=1)

In [3]:
def model_using_calibration_data():
    torch.manual_seed(0)
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    batch_size=32
    fs = 500
    train_ratio = 0.9
    sub = 7 #Till subject 7 (starting from 0), only calibration sessions are conducted

    # Xtr, Ytr = create_dataset(sub, base_path=parent_dir)
    Xtr, Ytr = utils.calib_sess_dataset(base_path=parent_dir)
    X_train = utils.baseline_correction(Xtr)
    X_train = utils.bandpass_filtering(X_train)

    # Creating train-validation split
    eeg_train, eeg_val, label_train, label_val = train_test_split(X_train, Ytr, 
                                                                train_size=train_ratio, random_state=42, shuffle=True)


    X_train_tensor = torch.tensor(eeg_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_train_tensor = torch.tensor(label_train, dtype=torch.long).to(device)  # Use long for classification

    X_val_tensor = torch.tensor(eeg_val, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_val_tensor = torch.tensor(label_val, dtype=torch.long).to(device)

    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)


    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    # optimizer = optim.Adam(model.parameters(), lr=1e-3)
    trained_model = train_utils.model_training(model, train_loader, val_loader)
    torch.save(trained_model.state_dict(), 'calibrated_eegnet_model.pth')  # Save best model

    return trained_model
    

In [4]:
def evaluation_on_online_data(model_file):
    torch.manual_seed(0)
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size=32

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model.load_state_dict(torch.load(f'{model_file}.pth'))

    online_perf = dict()
    for sub in range(8, 21):
        Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)

        test_loader = train_utils.convert_to_tensor(Xte, Yte)

        test_loss, test_acc = train_utils.model_evaluation(model, test_loader)
        online_perf[f'Sub{sub:02d}'] = test_acc
        
        print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

    print(f'Average Accuracy: {np.mean(list(online_perf.values()))}')
    print(list(online_perf.values()))

    return
    

In [ ]:
# Model Training on Calibration and Testing on Online Session Data
model_using_calibration_data()
evaluation_on_online_data('calibrated_eegnet_model')

In [6]:
# Model Fine Tuning
def model_fine_tuning_params(model_file, denseLayer=True, conv2dLayer=False):

    torch.manual_seed(0)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model.load_state_dict(torch.load(f'{model_file}.pth'))

    if denseLayer or conv2dLayer:

        for param in model.parameters():
            param.requires_grad = False

        if denseLayer:
            print(f'Dense layer is fine-tuned')
            # Freeze all layers except the last one
            for param in model.dense.parameters():
                param.requires_grad = True  # Freeze all parameters

        if conv2dLayer:
            print(f'Conv2D layer is fine-tuned')
            # Unfreeze the last layer parameters
            for param in model.separableConv.parameters():
                param.requires_grad = True  # Unfreeze last layer

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    return model, optimizer


In [7]:
def subject_specific_fine_tuning(denseLayer=True, conv2dLayer=False):
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    Xcalib, Ycalib = utils.calib_sess_dataset(base_path=parent_dir)

    train_ratio = 0.9
    batch_size=32
    online_perf = dict()

    for sub in range(8, 21):
        Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)
        eeg_train, eeg_val, label_train, label_val = train_test_split(Xtr, Ytr, 
                                                                train_size=train_ratio, random_state=42, shuffle=True)
        
        # Xtrain = np.concatenate((Xcalib, eeg_train), axis=0)
        # Ytrain = np.concatenate((Ycalib, label_train), axis=0)
        Xtrain = eeg_train
        Ytrain = label_train
        
        train_loader = train_utils.convert_to_tensor(Xtrain, Ytrain)
        val_loader = train_utils.convert_to_tensor(eeg_val, label_val)

        model, optimizer = model_fine_tuning_params('calibrated_eegnet_model', denseLayer=denseLayer, conv2dLayer=conv2dLayer)
        trained_model = train_utils.model_training(model, train_loader, val_loader, Tuning=True)

        test_loader = train_utils.convert_to_tensor(Xte, Yte)

        # model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
        # model.load_state_dict(torch.load('trained_model_checkpoint.pth'))

        test_loss, test_acc = train_utils.model_evaluation(trained_model, test_loader)

        print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')
        online_perf[f'Sub{sub:02d}'] = test_acc

    print(f'Average Accuracy: {np.mean(list(online_perf.values()))}')
    print(list(online_perf.values()))


In [ ]:
subject_specific_fine_tuning(denseLayer=False, conv2dLayer=False)

In [ ]:
subject_specific_fine_tuning(denseLayer=True, conv2dLayer=False)

In [ ]:
subject_specific_fine_tuning(denseLayer=True, conv2dLayer=True)